# 📋 [K14] 군 생활 데이터 분석 - 생활관 생활 데이터로 개선 과제 찾기
### [학생 실습용] 프롬프트 설계 실습 워크북 (Student Workbook)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/K14/K14_student.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 병영생활·부대관리 (Level 1) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 15.0시간) |
| **실습 유형** | 군 생활 데이터 통계 가공, HITL 수기 검산, 시각화 왜곡 방지 및 종합 보고서 |

### 📖 프로젝트 목표
생활관은 체력단련실·사이버지식정보방·PX 이용, 수면, 식단, 건의 같은 생활 데이터가 쌓이지만, 숫자를 표로만 두면 어디가 나아졌고 어디가 나빠지는지 알기 어렵습니다. 이 프로젝트에서는 가상 한빛대대 3개 중대의 4주간 생활관 생활 데이터를 생성형 AI로 요약하고 추세와 중대별 차이를 찾고, 그래프를 요청하고, 원인 가설과 개선 제안을 만든 뒤 발표 자료까지 정리합니다. AI가 계산한 수치를 직접 확인하는 습관과, 데이터만으로 원인을 단정하지 않는 태도를 함께 익힙니다.

> **⚠️ [교육용 가상 자료 및 보안 수칙 준수 원칙]**
> - 본 워크북의 모든 데이터는 장병 복지 증진을 위한 **교육용 가상 부대 데이터셋(가상 한빛대대)**입니다.
> - 실제 부대 현황, 장병 인적사항(성명, 군번 등) 및 비인가 군사자료는 절대 외부 AI 서비스에 입력하지 않습니다.
> - AI가 산출한 수치는 반드시 원문과 대조하여 검산(HITL)하며 최종 부대 개선 정책 결정은 지휘계통에 따릅니다.



In [ ]:
# ============================================================================
# [Colab 환경 설정 및 헬퍼 함수] 데이터 분석, 가상 데이터 로더 & 마크다운 렌더러
# ============================================================================
!pip install -q pandas numpy tabulate matplotlib seaborn

import pandas as pd
import numpy as np
import json
import os
import re
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

# 폰트 설정 (Colab / Linux 호환)
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial']
plt.rcParams['axes.unicode_minus'] = False

# ----------------------------------------------------------------------------
# [데이터셋 자동 로드] 가상 한빛대대 4주간 생활관 생활 데이터 (12행)
# ----------------------------------------------------------------------------
data_records = [
    {'주차': '1주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 112, '싸지방': 54, 'PX': 140, '수면만족': 3.8, '식단만족': 3.9, '건의': 3},
    {'주차': '1주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 98, '싸지방': 31, 'PX': 160, '수면만족': 3.6, '식단만족': 3.5, '건의': 4},
    {'주차': '1주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 104, '싸지방': 47, 'PX': 118, '수면만족': 3.7, '식단만족': 3.2, '건의': 5},
    {'주차': '2주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 118, '싸지방': 58, 'PX': 138, '수면만족': 3.8, '식단만족': 3.8, '건의': 2},
    {'주차': '2주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 92, '싸지방': 29, 'PX': 166, '수면만족': 3.4, '식단만족': 3.4, '건의': 5},
    {'주차': '2주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 106, '싸지방': 50, 'PX': 121, '수면만족': 3.6, '식단만족': 3.1, '건의': 7},
    {'주차': '3주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 121, '싸지방': 61, 'PX': 142, '수면만족': 3.7, '식단만족': 4.0, '건의': 4},
    {'주차': '3주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 85, '싸지방': 26, 'PX': 171, '수면만족': 3.2, '식단만족': 3.4, '건의': 7},
    {'주차': '3주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 103, '싸지방': 49, 'PX': 117, '수면만족': 3.7, '식단만족': 3.3, '건의': 9},
    {'주차': '4주차', '중대': '1중대', '인원(명)': 48, '체력단련실': 125, '싸지방': 66, 'PX': 135, '수면만족': 3.7, '식단만족': 4.1, '건의': 3},
    {'주차': '4주차', '중대': '2중대', '인원(명)': 52, '체력단련실': 80, '싸지방': 24, 'PX': 178, '수면만족': 3.0, '식단만족': 3.3, '건의': 8},
    {'주차': '4주차', '중대': '3중대', '인원(명)': 45, '체력단련실': 108, '싸지방': 52, 'PX': 120, '수면만족': 3.7, '식단만족': 3.2, '건의': 12}
]

df = pd.DataFrame(data_records)

def load_dataset(mission_num):
    """미션 번호(1~10)에 해당하는 가상 데이터셋 JSON을 로드합니다."""
    filename = f"k14_m{mission_num:02d}_dataset.json"
    for base in [".", "uploads/K14", "../uploads/K14"]:
        filepath = os.path.join(base, filename)
        if os.path.exists(filepath):
            with open(filepath, "r", encoding="utf-8") as f:
                return json.load(f)
    print(f"⚠️ 데이터셋 파일을 찾을 수 없습니다: {filename}")
    return None

def show_result(mission_title, prompt, result):
    """학습자가 입력한 프롬프트와 분석 결과 및 검증 리포트를 시각적으로 렌더링합니다."""
    p_text = prompt.strip() if prompt.strip() else "(작성된 프롬프트가 없습니다)"
    r_text = result.strip() if result.strip() else "(기록된 결과가 없습니다)"
    
    md_content = f"""
---
### 📌 [{mission_title}] 실습 기록 리포트

#### 💬 입력 프롬프트
```text
{p_text}
```

#### 📋 분석 결과 및 팩트체크 검증 내역
{r_text}
---
"""
    display(Markdown(md_content))

print("✅ [K14] 실습 환경, 12행 생활관 데이터셋(df) 및 시각화 렌더러 준비 완료!")
display(df)



### 📚 공통 가상 배경 및 데이터 모음

**K14 실습자료 - 군 생활 데이터 분석 - 생활관 생활 데이터로 개선 과제 찾기**

구    분 **실습자료 [교육용가상자료]**

**1. 사용 안내**

이 자료는 '군 생활 데이터 분석' 실습에 쓰는 교육용 가상 자료입니다. 가상 한빛대대의 이용 수·설문 점수·건의 건수는 모두 임의로 만든 수치로 실제 부대의 현황과 무관합니다. 모든 값은 중대 단위로 모은 집계 값이며 개인을 식별할 수 있는 정보가 없습니다. 생성형 AI에는 이 자료만 입력하고, 분석 결과는 참고용으로만 사용합니다.

**2. 가상 상황**

| **부대** | 가상 한빛대대 (교육용 가상 부대) |
| --- | --- |

| **상황** | 대대 복지 담당 병사인 학습자는 4주간의 생활관 생활 데이터를 보고, 중대장 회의에 낼 '생활관 개선 과제'를 정리하려 한다. |
| --- | --- |

| **원칙** | AI가 계산한 수치는 직접 확인한다. 원인은 가설로만 쓰고, 결과는 참고용이다. |
| --- | --- |

**3. 가상 생활관 생활 데이터 (4주 × 3개 중대)**

| **주차** | **중대** | **인원(명)** | **체력단련실 이용(회)** | **사이버지식정보방 이용(회)** | **PX 이용(회)** | **수면 만족(5점)** | **식단 만족(5점)** | **건의(건)** |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| 1주차 | 1중대 | 48 | 112 | 54 | 140 | 3.8 | 3.9 | 3 |
| 1주차 | 2중대 | 52 | 98 | 31 | 160 | 3.6 | 3.5 | 4 |
| 1주차 | 3중대 | 45 | 104 | 47 | 118 | 3.7 | 3.2 | 5 |
| 2주차 | 1중대 | 48 | 118 | 58 | 138 | 3.8 | 3.8 | 2 |
| 2주차 | 2중대 | 52 | 92 | 29 | 166 | 3.4 | 3.4 | 5 |
| 2주차 | 3중대 | 45 | 106 | 50 | 121 | 3.6 | 3.1 | 7 |
| 3주차 | 1중대 | 48 | 121 | 61 | 142 | 3.7 | 4.0 | 4 |
| 3주차 | 2중대 | 52 | 85 | 26 | 171 | 3.2 | 3.4 | 7 |
| 3주차 | 3중대 | 45 | 103 | 49 | 117 | 3.7 | 3.3 | 9 |
| 4주차 | 1중대 | 48 | 125 | 66 | 135 | 3.7 | 4.1 | 3 |
| 4주차 | 2중대 | 52 | 80 | 24 | 178 | 3.0 | 3.3 | 8 |
| 4주차 | 3중대 | 45 | 108 | 52 | 120 | 3.7 | 3.2 | 12 |

| 이용 수는 그 주에 중대 장병이 해당 시설을 이용한 총 횟수, 수면·식단 만족은 중대 설문 평균(1점 매우 불만 ~ 5점 매우 만족)입니다. |
| --- |

**4. 건의 유형별 건수 (4주 합계)**

| **중대** | **시설·환경** | **식단** | **일과·시간** | **기타** | **합계** |
| --- | --- | --- | --- | --- | --- |
| 1중대 | 4 | 3 | 3 | 2 | 12 |
| 2중대 | 6 | 4 | 10 | 4 | 24 |
| 3중대 | 17 | 9 | 3 | 4 | 33 |

**5. 특이사항 메모 (가설 세우기용)**

| **주차** | **대상** | **메모** |
| --- | --- | --- |
| 1주차 | 3중대 | 난방기 이상 소음 건의가 접수됨(점검 전) |
| 2주차 | 2중대 | 야간 경계 근무 교대 시간이 조정됨 |
| 3주차 | 전 중대 | 사격 훈련 준비로 개인정비 시간이 줄어듦 |
| 4주차 | 전 중대 | 동계 피복 보급 완료, 3중대 난방기 점검 예정 공지 |

| 메모는 원인을 확정하는 근거가 아니라 가설을 세울 때 참고하는 단서입니다. |
| --- |

**미션별 작성 서식**


---

## 🎯 미션 01. 생활관 데이터 항목 이해하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **생활관 생활 데이터의 항목과 의미를 이해합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **실습자료의 4주간 중대별 데이터 표에서 항목(체력단련실·사이버지식정보방·PX 이용 수, 수면 만족 점수, 식단 만족도, 건의 건수)의 단위와 뜻을 정리하고, 분석으로 알고 싶은 질문 3개를 정 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `데이터 항목 설명표, 분석 질문 3개, 개인 식별 정보 점검 결과` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 1 데이터 항목 설명표

| **항목** | **단위** | **뜻** | **분석에 쓸 수 있는 질문** |
| --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 문제 정의 — 데이터 항목을 이해하고 분석 질문과 개인정보 점검을 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(데이터 항목 설명표, 분석 질문 3개, 개인 식별 정보 점검 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 01.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_01_prompt = """
# [TODO] 생활관 6대 지표의 단위·의미 정립 및 핵심 질문 3개를 질의하는 프롬프트를 작성하세요.
# 예시: 6개 지표(체력, 싸지방, PX, 수면, 식단, 건의)의 단위 및 관리 목적, 12행 데이터 기반 3대 질문 도출
"""

mission_01_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 01. 생활관 데이터 항목 이해하기", mission_01_prompt, mission_01_result)



---

## 🎯 미션 02. 데이터 표 정리하고 점검하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에 넣을 데이터를 표로 정리하고 이상한 값을 점검합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **12행(4주 × 3개 중대)의 표를 AI가 읽기 쉬운 형태로 정리하고, 빠진 값·단위 혼동·범위를 벗어난 값이 없는지 점검 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `정리한 데이터 표, 데이터 점검 기록` 완성

### 📑 실습 서식 및 대조 점검 양식
실습가이드 지침에 따라 가상 데이터를 분석하고 프롬프트/점검표를 완성합니다.

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 정보 구조화 — 데이터를 읽기 쉬운 표로 정리하고 이상한 값과 비교 기준을 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(정리한 데이터 표, 데이터 점검 기록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 02.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_02_prompt = """
# [TODO] 12행 데이터를 표준 표로 정형화하고 무결성(결측치, 1~5점 범위) 점검을 요청하는 프롬프트를 작성하세요.
# 예시: 12행 표 출력 및 중대별 인원 차이에 따른 1인당 계산 계획 수립 요청
"""

mission_02_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 02. 데이터 표 정리하고 점검하기", mission_02_prompt, mission_02_result)



In [ ]:
# [미션 02 실습 보조] 결측치 및 데이터 무결성 검증 (Pandas)
print('--- 결측치(Null) 점검 ---')
print(df.isnull().sum())

print('\n--- 만족도 점수 범위 점검 (1~5점 척도 여부) ---')
print(df[['수면만족', '식단만족']].describe().loc[['min', 'max']])

print('\n--- 1인당 계산용 중대별 정원 현황 ---')
display(df.groupby('중대')['인원(명)'].first().to_frame())

---

## 🎯 미션 03. 데이터 요약 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI에게 데이터 요약을 요청합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **데이터 표를 붙이고 '항목별 4주 합계·평균과 가장 높거나 낮은 중대를 표로 정리하라'는 요약 프롬프트를 설계하고 작성하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `요약 프롬프트, AI 요약 결과` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 3 데이터 요약 프롬프트 틀

| 너는 가상 한빛대대의 생활 데이터를 쉽게 설명하는 데이터 분석 도우미다.
아래 [데이터]만 사용하고, 없는 값은 만들지 마라.
항목별로 4주 합계와 주간 평균, 가장 높은 중대와 낮은 중대를 표로 정리하라.
계산 과정을 한 줄로 함께 보여라.
[데이터]
(데이터 표 붙여 넣기) |
| --- |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 프롬프트 작성(분석 요청) — 데이터를 제공하고 요약 항목과 형식을 지정해 분석을 요청했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(요약 프롬프트, AI 요약 결과)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 03.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_03_prompt = """
# [TODO] 6개 항목별 4주 합계, 주간 평균, 최고/최저 중대 요약을 요청하는 프롬프트를 작성하세요.
# 예시: [요구사항]을 구체적으로 명시하고 마크다운 표 출력을 지시하는 프롬프트 작성
"""

mission_03_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 03. 데이터 요약 요청하기", mission_03_prompt, mission_03_result)



---

## 🎯 미션 04. 요약 수치 직접 확인하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **AI가 계산한 수치를 직접 확인합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **AI가 알려 준 합계·평균·증감률을 계산기나 표 계산 도구로 다시 계산해 맞음·틀림으로 검토하고 핵심 정보를 식별하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `수치 검산표, 오류 원인과 수정 요청` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 4 수치 검산표

| **항목·중대** | **AI가 알려 준 값** | **직접 계산한 값** | **맞음·틀림** | **오류 원인** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] AI 결과 검증 — AI가 계산한 수치를 직접 다시 계산해 오류를 찾고 바로잡았는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(수치 검산표, 오류 원인과 수정 요청)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 04.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_04_prompt = """
# [TODO] AI가 계산한 합계 수치를 직접 수기 검산하고 오류 발생 시 CoT 재질의를 요청하는 프롬프트를 작성하세요.
# 예시: 원문 데이터와 AI 산출값 1:1 대조 및 단계별 덧셈 유도 피드백 프롬프트
"""

mission_04_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 04. 요약 수치 직접 확인하기", mission_04_prompt, mission_04_result)



In [ ]:
# [미션 04 실습 보조] AI 산출 수치 직접 1:1 검산 (HITL 연산 검증)
print('=== 중대별 4주간 총합 (체련실/싸지방/PX/건의) ===')
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
display(totals)

print('\n=== 중대별 4주간 만족도 평균 (수면/식단) ===')
averages = df.groupby('중대')[['수면만족', '식단만족']].mean().round(2)
display(averages)

# AI 요약 결과와 대조하여 1:1 맞음/틀림을 판정하세요.

---

## 🎯 미션 05. 주차별 추세 파악하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **주차별 변화 추세를 파악합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중대별로 1주차에서 4주차까지 늘었는지, 줄었는지, 비슷한지 AI에게 정리하게 하고 증감폭을 직접 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `중대별 주차 추세표, 추세 해석 메모` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 5 중대별 주차 추세표

| **중대** | **항목** | **1주차** | **4주차** | **증감** | **추세처럼 보임 / 확정 어려움** |
| --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 추세 분석 — 주차별 변화를 증감폭으로 정리하고 짧은 기간의 한계를 구분했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(중대별 주차 추세표, 추세 해석 메모)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 05.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_05_prompt = """
# [TODO] 1주차 대비 4주차 증감폭/증감률 계산 및 추세 해석 메모 작성을 요청하는 프롬프트를 작성하세요.
# 예시: 수면 만족도 하락, 건의 급증 등 추세 방향 분석 및 '단정 방지' 문구 포함 요청
"""

mission_05_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 05. 주차별 추세 파악하기", mission_05_prompt, mission_05_result)



---

## 🎯 미션 06. 중대별 비교하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **중대별 차이를 공정하게 비교합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **인원이 다른 3개 중대를 총합이 아니라 1인당 값으로 비교하게 하고, 총합과 1인당 값의 순위가 달라지는 항목이 있는지 확인하고 유효성을 점검하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `중대별 1인당 비교표, 주목할 항목 목록` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 6 1인당 비교표

| **항목** | **1중대(1인당)** | **2중대(1인당)** | **3중대(1인당)** | **총합 순위와 달라졌는가** |
| --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 비교 분석 — 인원 차이를 고려한 기준으로 중대별 차이를 비교하고 주목 항목을 골랐는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(중대별 1인당 비교표, 주목할 항목 목록)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 06.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_06_prompt = """
# [TODO] 중대 인원수를 나눈 1인당 이용량을 산출하고 순위 역전을 비교하는 프롬프트를 작성하세요.
# 예시: 1인당 체력단련실, 싸지방, PX 이용량 산출 및 총합 순위와의 차이 비교
"""

mission_06_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 06. 중대별 비교하기", mission_06_prompt, mission_06_result)



In [ ]:
# [미션 06 실습 보조] 중대별 1인당 4주 총 이용 횟수 계산 (정규화)
personnel = df.groupby('중대')['인원(명)'].first()
totals = df.groupby('중대')[['체력단련실', '싸지방', 'PX', '건의']].sum()
per_capita = totals.div(personnel, axis=0).round(2)

print('=== 1인당 4주 총 이용 횟수 (총합 ÷ 인원수) ===')
display(per_capita[['체력단련실', '싸지방', 'PX']])

print('\n=== 장병 100명당 건의 발생률 ===')
suggestions_rate = (totals['건의'].div(personnel) * 100).round(1).to_frame(name='100명당 건의 건수')
display(suggestions_rate)

---

## 🎯 미션 07. 그래프 시각화 요청하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **그래프 시각화를 요청하고 오해를 막습니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **추세에는 선 그래프, 중대 비교에는 막대 그래프처럼 목적에 맞는 그래프를 AI에게 요청 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `그래프 요청 프롬프트 2개, 그래프 점검표` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 7 그래프 요청 프롬프트 틀과 점검표

| 위 [데이터]로 (선 그래프 / 막대 그래프)를 만들어 줘.
- 그릴 항목: 
- 가로축: 
- 세로축 이름과 단위: 
- 세로축은 0부터 시작하게 하거나 시작값을 명시해 줘.
- 제목과 범례를 넣어 줘.
그래프를 그릴 수 없으면 표 계산 도구에서 만드는 방법을 단계별로 알려 줘. |
| --- |

| **점검 항목** | **그래프 1** | **그래프 2** |
| --- | --- | --- |
| 축 이름과 단위가 맞다 | ( ) | ( ) |
| 축 시작값이 과장되지 않았다 | ( ) | ( ) |
| 값이 표와 일치한다 | ( ) | ( ) |
| 제목이 내용을 설명한다 | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 시각화 요청·검증 — 목적에 맞는 그래프를 요청하고 축과 단위를 데이터와 대조해 점검했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(그래프 요청 프롬프트 2개, 그래프 점검표)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 07.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_07_prompt = """
# [TODO] 추세 선 그래프(수면만족)와 비교 막대 차트(1인당 체련실) 생성 프롬프트 2종을 작성하세요.
# 예시: 축 이름, 단위, Y축 시작값(0부터 시작) 명시 및 왜곡 점검 체크리스트
"""

mission_07_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 07. 그래프 시각화 요청하기", mission_07_prompt, mission_07_result)



In [ ]:
# [미션 07 실습 보조] AI 생성 시각화 코드 실행 및 왜곡(축/스케일) 점검
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. 추세 선 그래프 (수면 만족도)
for comp in ['1중대', '2중대', '3중대']:
    sub = df[df['중대'] == comp]
    axes[0].plot(sub['주차'], sub['수면만족'], marker='o', linewidth=2, label=comp)
axes[0].set_title('주차별 수면 만족도 변화 추이 (Line Chart)', fontsize=13)
axes[0].set_ylim(0, 5)  # 5점 척도 전체 표시 (왜곡 방지)
axes[0].set_ylabel('만족도 점수 (1~5점)')
axes[0].grid(True, linestyle='--', alpha=0.5)
axes[0].legend()

# 2. 비교 막대 그래프 (1인당 체력단련실 이용)
comp_names = ['1중대', '2중대', '3중대']
personnel = df.groupby('중대')['인원(명)'].first()
gym_per_capita = df.groupby('중대')['체력단련실'].sum().div(personnel).loc[comp_names]
bars = axes[1].bar(comp_names, gym_per_capita, color=['#3366cc', '#dc3912', '#109618'], width=0.5)
axes[1].set_title('중대별 1인당 4주 체력단련실 이용 횟수 (Bar Chart)', fontsize=13)
axes[1].set_ylim(0, 12)  # Y축 0부터 시작 (정직한 시각화)
axes[1].set_ylabel('1인당 평균 이용 횟수(회)')
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

# 막대 상단 수치 표기
for bar in bars:
    h = bar.get_height()
    axes[1].annotate(f'{h:.2f}회', xy=(bar.get_x() + bar.get_width()/2, h),
                     xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

---

## 🎯 미션 08. 원인 가설과 개선 제안 만들기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **원인 가설과 개선 제안을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **주목할 항목마다 데이터와 특이사항 메모를 근거로 원인 가설을 AI와 함께 2~3개 세우고, 각 가설을 확인하려면 어떤 추가 자료가 필요한지 적습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `원인 가설 표, 추가 확인 자료, 개선 제안 우선순위` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 8 원인 가설 표

| **주목 항목** | **가설** | **근거(데이터·메모)** | **추가로 확인할 자료** | **개선 제안** | **우선순위** |
| --- | --- | --- | --- | --- | --- |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |
| ( ) | ( ) | ( ) | ( ) | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 가설·제안 도출 — 근거 있는 가설과 확인 방법, 현실적인 개선 제안을 정리했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(원인 가설 표, 추가 확인 자료, 개선 제안 우선순위)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 08.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_08_prompt = """
# [TODO] 특이사항 메모와 건의 유형 통계를 연계하여 다각도 원인 가설 2~3개와 개선 제안을 요청하세요.
# 예시: 3중대 난방기 소음 가설, 2중대 야간근무 가설 수립 및 우선순위별 개선안 도출
"""

mission_08_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 08. 원인 가설과 개선 제안 만들기", mission_08_prompt, mission_08_result)



---

## 🎯 미션 09. 발표 자료 구성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **발표 자료의 구성을 만듭니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **결론 먼저 말하는 5장 구성(핵심 요약, 현황, 원인 가설, 개선 제안, 확인할 점)을 AI와 함께 잡고 장마다 한 줄 핵심 문장과 사용할 그래프를 정 수행하고 검증하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `발표 자료 5장 구성안` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 9 발표 자료 5장 구성

| **장** | **제목** | **한 줄 핵심 문장** | **쓸 그래프** |
| --- | --- | --- | --- |
| 1 | 핵심 요약 | ( ) | ( ) |
| 2 | 현황 | ( ) | ( ) |
| 3 | 원인 가설 | ( ) | ( ) |
| 4 | 개선 제안 | ( ) | ( ) |
| 5 | 확인할 점 | ( ) | ( ) |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [선택] 대상 맞춤 활용 — 청중에게 맞는 순서와 쉬운 표현으로 발표 자료를 구성했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(발표 자료 5장 구성안)의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 09.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_09_prompt = """
# [TODO] 지휘관 보고를 위한 결론 우선 5장 슬라이드 기획안과 1줄 헤드라인 작성을 요청하세요.
# 예시: 1장 핵심요약부터 5장 결심사항까지 두괄식 구조 설계
"""

mission_09_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 09. 발표 자료 구성하기", mission_09_prompt, mission_09_result)



---

## 🎯 미션 10. 생활관 개선 과제 보고서 완성하기
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 지침
#### 미션 배경 및 당위성

실제 부대 실무 현장에서는 불명확한 정보 전달이나 수동 업무 처리로 인해 불필요한 행정 소요가 발생하고 주요 공지·규정 확인이 지연되는 문제가 발생하기 쉽습니다. 따라서 **생활관 개선 과제 보고서를 완성합니다.** 과업을 체계적으로 수행하여 현장 업무 효율성을 극대화하고 정보 누락으로 인한 혼선을 사전에 방지해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **데이터 요약, 검산 결과, 추세·비교, 그래프, 가설과 개선 제안, 발표 구성을 한 편의 보고서로 묶습니다 과정을 진행하는** 실습을 단계별로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동을 순차적으로 진행하며 핵심 산출물을 도출하고 검증 기준을 달성합니다.

**이번 미션 결과물:** `생활관 개선 과제 보고서(최종본)` 완성

### 📑 실습 서식 및 대조 점검 양식
미션 10 보고서 목차

| 1. 분석 목적과 데이터 설명
2. 요약과 검산 결과
3. 추세와 중대별 비교
4. 그래프
5. 원인 가설과 개선 제안
6. 분석의 한계 (기간 4주, 집계 값, 가설)
7. 안내 문구: 가상 데이터 분석 결과이며 참고용, 실제 결정은 지휘계통 |
| --- |

**채점표**

| **미션** | **평가 항목** | **구분** | **판정** |
| --- | --- | --- | --- |
| 1 | 생활관 데이터 항목 이해하기 | 필수 | PASS / FAIL |
| 2 | 데이터 표 정리하고 점검하기 | 필수 | PASS / FAIL |
| 3 | 데이터 요약 요청하기 | 필수 | PASS / FAIL |
| 4 | 요약 수치 직접 확인하기 | 필수 | PASS / FAIL |
| 5 | 주차별 추세 파악하기 | 필수 | PASS / FAIL |
| 6 | 중대별 비교하기 | 선택 | PASS / FAIL |
| 7 | 그래프 시각화 요청하기 | 필수 | PASS / FAIL |
| 8 | 원인 가설과 개선 제안 만들기 | 필수 | PASS / FAIL |
| 9 | 발표 자료 구성하기 | 선택 | PASS / FAIL |
| 10 | 생활관 개선 과제 보고서 완성하기 | 필수 | PASS / FAIL |

### 🔍 평가 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가 기준:** [필수] 결과물 완성 — 분석 과정과 개선 제안을 정리하고 한계와 확인 안내를 포함해 완성했는가? (PASS / FAIL)

#### 미션 PASS 조건

- 요구 산출물(생활관 개선 과제 보고서(최종본))의 완성도 충족


- 원문 대조 점검을 통한 사실 검증 및 정확성 확보


- 교육용 가상 데이터 활용 및 군사보안 수칙 엄수


#### 채점자 판정 메모

> 단순 결과물 생성 여부뿐만 아니라 AI 결과물의 오류를 직접 찾아내고 바로잡았는지 대조표를 확인한다.



In [ ]:
# [미션 10.] 학생 실습 - 프롬프트 작성 및 결과 기록
mission_10_prompt = """
# [TODO] 미션 1~9의 모든 분석 결과와 거버넌스 고지문을 포함한 종합 보고서 작성을 요청하세요.
# 예시: Ⅰ.개요 ~ Ⅳ.개선과제, 분석 한계 및 지휘관 결심 안내 문구 포함
"""

mission_10_result = """
# [TODO] AI 질의 응답 결과 및 실습 서식에 따른 분석/검산 기록을 여기에 작성하세요.
"""

# 렌더러 실행 (작성한 프롬프트와 분석 결과를 렌더링합니다)
show_result("미션 10. 생활관 개선 과제 보고서 완성하기", mission_10_prompt, mission_10_result)

